### Extração de características dos dados do Benchmark Dataset (Tsinghua Lab) para o participante S1

Este notebook lê as epochs filtradas do participante S1, extrai características no demínio do tempo e da frequência de cada canal selecionado e salva o vetor de características e seus rótulos para a etapa de classificação.

**Características**

Inicialmente, foram usadas as nove características apresentadas em aula. Em que, nas fórmulas, N é o número de amostrar do sinal e M é o número de pontos do espectro. As fórmulas foram retiradas do artigo \[1\] Phinyomark, Angkoon & Limsakul, Chusak & Phukpattaranont, P.. (2008). EMG Feature Extraction for Tolerance of White Gaussian Noise.
[Disponível neste link](https://www.researchgate.net/publication/263765853_EMG_Feature_Extraction_for_Tolerance_of_White_Gaussian_Noise)

Domínio do tempo

1. Willison Amplitude (WAMP)

    > $ \sum_{i=1}^{N-1}f(|x_i - x_{i+1}|) $

    > $ f(x) = \begin{cases} 1 & \text{if } x \gt threshold \\ 0 & \text{otherwise} \end{cases} $

Segundo Phinyomark et al. (2009), a WAMP é o número de vezes em que a diferença de amplitude entre duas amostras adjacentes ultrapassa um limiar predefinido. O limiar existe para reduzir o efeito do ruído, assim como no ZC e no SSC. Aplicada ao EEG, a ideia é medir quantas variações rápidas e relevantes o sinal tem, o que depende do quanto o sinal "salta" entre amostras vizinhas; e no SSVEP, a WAMP permite avaliar quantas variações significativas de amplitude ocorrem no sinal EEG. Matematicamente, é a soma de indicadores $ f(|x_i - x_{i+1}|) \in \{0, 1\} $: uma contagem de quantas vezes a derivada discreta do sinal supera um valor fixo, o que torna a escolha do limiar (aqui, 1 µV) determinante para o resultado.

2. Variance of EMG (VAR)

    > $ \frac{1}{N-1}\sum_{i=1}^{N}x_i^2 $

A VAR mede a dispersão dos valores do sinal em relação à média, buscando identificar aumentos na amplitude da atividade cerebral pela maior potência do sinal. No EEG, permite avaliar a amplitude eficaz da atividade elétrica em cada canal. No paradigma SSVEP, valores mais elevados podem indicar uma resposta de maior intensidade à estimulação, especialmente nos canais occipitais. Matematicamente, é o segundo momento do sinal, ou seja, a potência média, que pelo teorema de Parseval é proporcional à soma do espectro de potência sobre todas as frequências.

3. Root Mean Square (RMS)

    > $ \sqrt{\frac{1}{N}\sum_{i=1}^{N}|x_i|^2} $

O RMS mede a amplitude eficaz e a intensidade da atividade do sinal em cada canal, indicando sua potência média. Sua aplicação pode mostrar o mesmo que a característica VAR, tanto no EEG quanto no SSVEP. Matematicamente, corresponde à raiz quadrada da média dos valores ao quadrado. Está diretamente relacionado à variância, pois o RMS ao quadrado é aproximadamente igual à VAR quando a média do sinal é próxima de zero.

4. Waveform Length (WL)

    > $ \sum_{i=1}^{N-1}|x_{i+1} - x_i| $

A WL mede o comprimento acumulado da forma de onda ao longo do tempo, refletindo a amplitude, a frequência e a complexidade das oscilações do sinal. No EEG, a WL permite avaliar a quantidade de variação acumulada no sinal ao longo do tempo. E no paradigma SSVEP, pode refletir mudanças na amplitude e na frequência das oscilações cerebrais provocadas pela estimulação visual.  Matematicamente, corresponde à soma das diferenças absolutas entre amostras consecutivas, representando a variação total do sinal.

5. Zero Crossing (ZC)

    > $ \sum_{i=1}^{N-1}sgn(x_i) $

    > $ sgn(x) = \begin{cases} 1 & \text{if } x_i * x_{i+1} \leq 0 \\ 0 & \text{otherwise} \end{cases} $

O ZC contabiliza quantas vezes o sinal cruza o zero, fornecendo uma estimativa indireta da frequência das oscilações. No EEG, pode ajudar a identificar mudanças na frequência da atividade cerebral. No paradigma SSVEP, pode indicar diferenças na frequência das oscilações provocadas pela estimulação visual, relacionadas às frequências de estimulação dos alvos. Entretanto, não identifica diretamente a frequência do estímulo, pois considera todos os cruzamentos do sinal, inclusive aqueles provocados por ruídos. Matematicamente, corresponde à contagem de mudanças de sinal entre amostras consecutivas. Nesta implementação, não é utilizado limiar para desconsiderar pequenas variações.

Domínio da frequência

1. Median Frequency (FMD)

    > $ \frac{1}{2}\sum_{j=1}^{M}PSD_j $

A FMD identifica a frequência que divide o espectro de potência em duas regiões de mesma potência acumulada. No EEG, permite avaliar como a energia do sinal está distribuída entre as frequências. No paradigma SSVEP, pode indicar a região de frequência que concentra a maior parte da potência cerebral, ajudando a observar mudanças na resposta associadas à estimulação visual. Entretanto, por considerar todo o espectro, não identifica necessariamente a frequência do estímulo. Matematicamente, corresponde à frequência na qual a potência acumulada atinge 50% da potência total do espectro.

2. Mean Frequency (FMN)

    > $ \sum_{j=1}^{M}f_j PSD_j \Big/ \sum_{j=1}^{M}PSD_j $

    > $ f_j = \frac{j \cdot SampleRate}{2M} $

A FMN representa a frequência média ponderada pela potência do espectro, funcionando como seu centroide. No EEG, permite identificar em qual região de frequência a atividade elétrica apresenta maior concentração de potência. No paradigma SSVEP, pode indicar deslocamentos na distribuição espectral associados às respostas evocadas pela estimulação visual. Entretanto, como considera todo o espectro, a frequência característica do SSVEP pode ter sua influência diluída pelas demais componentes. Matematicamente, corresponde à soma das frequências multiplicadas por suas respectivas potências, dividida pela potência total do espectro.

3. Modified Median Frequency (MMDF)

    > $ \frac{1}{2}\sum_{j=1}^{M}A_j $

    > $ A_j = Amplitude\ do\ espectro\ j $

A MMDF identifica a frequência que divide o espectro de amplitude em duas regiões de mesma soma acumulada. No EEG, permite avaliar a distribuição das amplitudes entre as frequências. No paradigma SSVEP, pode ajudar a identificar mudanças na distribuição espectral associadas à resposta cerebral provocada pelo estímulo visual. Diferentemente da FMD, utiliza o espectro de amplitude em vez do espectro de potência, apresentando maior tolerância ao ruído nos experimentos do artigo. Matematicamente, corresponde à frequência na qual a soma acumulada das amplitudes atinge 50% da amplitude espectral total.

4. Modified Frequency Mean (MMNF)

    > $ \sum_{j=1}^{M}f_jA_j \Big/ \sum_{j=1}^{M}A_j $

A MMNF representa a frequência média ponderada pelas amplitudes do espectro, funcionando como seu centroide. No EEG, permite identificar a região de frequência em torno da qual as amplitudes espectrais estão concentradas. No paradigma SSVEP, pode ajudar a observar mudanças na distribuição das respostas cerebrais entre diferentes frequências de estimulação. Segundo o artigo, apresentou elevada tolerância ao ruído, embora isoladamente tenha baixa capacidade de discriminar as classes. Matematicamente, corresponde à soma das frequências multiplicadas por suas respectivas amplitudes, dividida pela soma total das amplitudes espectrais.

In [34]:
# Carregar as epochs filtradas

import mne
import numpy as np

epochs = mne.read_epochs('data/output/S1_filtered_epo.fif', preload=True, verbose=False)
fs = epochs.info['sfreq']

print(epochs)

<EpochsFIF | 240 events (all good), -0.5 – 5.496 s (baseline off), ~175.8 MiB, data loaded, with metadata,
 '8.0': 6
 '9.0': 6
 '10.0': 6
 '11.0': 6
 '12.0': 6
 '13.0': 6
 '14.0': 6
 '15.0': 6
 '8.2': 6
 '9.2': 6
 and 30 more events ...>


Foram selecionados os nove eletrodos usados na avaliação com FBCCA do artigo referência (Pz, PO5, PO3, POz, PO4, PO6, O1, Oz e O2). Além disso, foi usado um recorte de 0 a 5 segundos de cada epoch, para descartar os 0,5 segundos antes do estímulo. Por fim, o sinal foi convertido de volts para microvolts, que é a unidade utilizada pelo artigo.

In [35]:
p_occipital = ['PZ', 'PO5', 'PO3', 'POZ', 'PO4', 'PO6', 'O1', 'OZ', 'O2']
selected_channels = [name for name in epochs.ch_names if name.upper() in p_occipital]
print(len(selected_channels), selected_channels)

# Seleciona os canais e recorta o período de estímulo
epochs_selected = epochs.copy().pick(selected_channels).crop(tmin=0, tmax=5)

x = epochs_selected.get_data() * 1e6
y = epochs_selected.events[:, 2]

print(x.shape)
print(y.shape)

9 ['PZ', 'PO5', 'PO3', 'POz', 'PO4', 'PO6', 'O1', 'Oz', 'O2']
(240, 9, 1251)
(240,)


**Implementação de características**

Foram implementadas cara característica proposta sobre o último eixo do array, de modo que uma única chamada calcula a característica de todas as epochs e de todos os canais. 

No domínio do tempo, o WAMP depende de um limiar, decidi utilizar 1 µV. VAR e RMS foram divididos pelo número de amostrar de cada sinal, como nas fórmulas.

Já no domínio da frequência, os sinais foram transformados com a FFT, que devolve apenas as frequências positivas, de 0 até a metade da frequência de amostragem; e foi calculado o PSD como o quadrado do módulo do espectro, enquanto as versões modificadas usam a amplitude.

In [36]:
# funções auxiliares
def PSD(w):
    ''' definição da função PSD para o sinal no domínio da frequência '''
    return np.abs(w) ** 2

# Número de vezes em que a diferença entre amostras vizinhas passa do limiar
def wamp(z, threshold):
    return np.sum(np.abs(np.diff(z, axis=-1)) > threshold, axis=-1)


# Soma dos quadrados dividida por N - 1
def var(z):
    return np.sum(z ** 2, axis=-1) / (np.prod(z.shape) - 1)


# Raiz da média dos quadrados
def rms(z):
    return np.sqrt(np.sum(np.abs(z) ** 2, axis=-1) / (np.prod(z.shape) - 1))


# Soma das variações absolutas entre amostras vizinhas
def wl(z):
    return np.sum(np.abs(np.diff(z, axis=-1)), axis=-1)


# Número de vezes em que amostras vizinhas têm sinais opostos ou nulos
def zc(z):
    return np.sum(z[..., :-1] * z[..., 1:] <= 0, axis=-1)


# PSD de um espectro
def psd(w):
    return np.abs(w) ** 2


# Metade da soma do PSD
def fmd(w):
    return np.sum(PSD(w), axis=-1) / 2


# Média das frequências ponderada pelo PSD
def fmn(w, freqs):
    return np.sum(freqs * psd(w), axis=-1) / np.sum(psd(w), axis=-1)


# Metade da soma das amplitudes
def mmdf(w):
    return np.sum(np.abs(w), axis=-1) / 2


# Média das frequências ponderada pela amplitude
def mmnf(w, freqs):
    return np.sum(freqs * np.abs(w), axis=-1) / np.sum(np.abs(w), axis=-1)

Para extrair as características, primeiro foi calculado o espectro de cada sinal e o eixo de frequências correspondente, e depois aplicado as funções aos dados.

In [37]:
threshold = 1.0

spectrum = np.fft.rfft(x, axis=-1)
freqs = np.fft.rfftfreq(x.shape[-1], d=1 / fs)

feature_list = [
    wamp(x, threshold),
    var(x),
    rms(x),
    wl(x),
    zc(x),
    fmd(spectrum),
    fmn(spectrum, freqs),
    mmdf(spectrum),
    mmnf(spectrum, freqs)
]

features = np.array(feature_list)
print(features.shape)

(9, 240, 9)


O vetor de características tem uma linha por epoch e, em cada linha, as novas características de um canal em sequência, seguidas pelas do canal seguinte. Por isso, o array de características foi reordenado para [epoch, canal, característica] e depois para duas dimensões. Os rótulos são o código do alvo de cada epoch, de 0 a 39.

In [ ]:
features_ordered = features.transpose(1, 2, 0)
print(features_ordered.shape)

x_final = features_ordered.reshape(features_ordered.shape[0], features_ordered.shape[1] * features_ordered.shape[2])
print(x_final.shape)

np.save('data/output/S1_features.npy', x_final)
np.save('data/output/S1_labels.npy', y)

(240, 9, 9)
(240, 81)


**Características do Canonical Correlation Analysis**

O artigo referência \[1\] Yijun Wang, Xiaogang Chen, Xiaorong Gao, Shangkai Gao (2017). A Benchmark Dataset for SSVEP-Based Brain–Computer Interfaces ([https://ieeexplore.ieee.org/document/7740878](https://ieeexplore.ieee.org/document/7740878)) também aponta o uso do método FBCCA como um conjunto de característica utilizado para classificação. Porém, por sua complexidade, decidi utilizar o método mais simples em que este se baseia, o CCA. 

A CCA é um método estatístico que identifica a relação entre dois conjuntos de sinais, buscando combinações lineares de suas variáveis que apresentem a maior correlação possível. No contexto do EEG, compara os sinais cerebrais registrados com sinais de referência construídos a partir de frequências específicas, permitindo identificar a presença de respostas SSVEP.

Foram usados os mesmo nove canais e o mesmo recorte de 0 a 5 segundos (x). para cada epoch e para cada alvo $k$, a característica é a correlação canônica $ \rho_k $ entre o sinal dos nove canais e os sinais de referência de seno e cosseno do alvo $k$:

> $ Y_k = \left[ \sin(2\pi h f_k t),\ \cos(2\pi h f_k t) \right],\ h = 1, \dots, N_h $

\[1\] Xiaogang Chen, Yijun Wang, Shangkai Gao, Tzyy-Ping Jung and Xiaorong Gao (2015). Filter bank canonical correlation analysis for implementing a high-speed SSVEP-based
brain–computer interface ([https://pubmed.ncbi.nlm.nih.gov/26035476/](https://pubmed.ncbi.nlm.nih.gov/26035476/))

Como as epochs filtradas só mantêm de 6 a 25Hz, foram usados $N_h = 2$ harmônicos (fundamental e segundo harmônico). Assim, cada epoch tem 40 características, uma por alvo.

In [39]:
# Frequência de cada alvo, na ordem do código do alvo
target_freqs = np.zeros(len(epochs.event_id))
for name, target in epochs.event_id.items():
    target_freqs[target] = float(name)

n_harmonics = 2
t = np.arange(x.shape[-1]) / fs

# Sinais de referência de cada alvo, centralizados e ortonormalizados
reference_q = []
for f in target_freqs:
    ref = np.column_stack([fn(2 * np.pi * h * f * t) for h in range(1, n_harmonics + 1) for fn in (np.sin, np.cos)])
    ref = ref - ref.mean(axis=0)
    reference_q.append(np.linalg.qr(ref)[0])

# Maior correlação canônica entre cada epoch e cada alvo
cca_features = np.zeros((x.shape[0], len(target_freqs)))
for e in range(x.shape[0]):
    xe = x[e] - x[e].mean(axis=1, keepdims=True)
    qx = np.linalg.qr(xe.T)[0]
    for k in range(len(target_freqs)):
        cca_features[e, k] = np.linalg.svd(qx.T @ reference_q[k], compute_uv=False)[0]

print(cca_features.shape)

np.save('data/output/S1_cca_features.npy', cca_features)

(240, 40)
